In [ ]:
import numpy as np
import math
import random
import matplotlib.pyplot as plt
!pip install line_profiler
%load_ext line_profiler
!pip install ipython-autotime
%load_ext autotime


Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 67 kB 3.1 MB/s 
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 1.6 MB 5.1 MB/s 
time: 856 µs (started: 2022-09-22 03:53:05 +00:00)


In [ ]:
S_BOX = {
  "0x00": "63", "0x01": "7C", "0x02": "77", "0x03": "7B", "0x04": "F2", "0x05": "6b", "0x06": "6f", "0x07": "c5",
  "0x08": "30", "0x09": "01", "0x0a": "67", "0x0b": "2b", "0x0c": "fe", "0x0d": "d7", "0x0e": "ab", "0x0f": "76", 

  "0x10": "ca", "0x11": "82", "0x12": "c9", "0x13": "7d", "0x14": "fa", "0x15": "59", "0x16": "47", "0x17": "f0",
  "0x18": "ad", "0x19": "d4", "0x1a": "a2", "0x1b": "af", "0x1c": "9c", "0x1d": "a4", "0x1e": "72", "0x1f": "c0",

  "0x20": "b7", "0x21": "fd", "0x22": "93", "0x23": "26", "0x24": "36", "0x25": "3f", "0x26": "f7", "0x27": "cc",
  "0x28": "34", "0x29": "a5", "0x2a": "e5", "0x2b": "f1", "0x2c": "71", "0x2d": "d8", "0x2e": "31", "0x2f": "15",

  "0x30": "04", "0x31": "c7", "0x32": "23", "0x33": "c3", "0x34": "18", "0x35": "96", "0x36": "05", "0x37": "9a",
  "0x38": "07", "0x39": "12", "0x3a": "80", "0x3b": "e2", "0x3c": "eb", "0x3d": "27", "0x3e": "b2", "0x3f": "75",

  "0x40": "09", "0x41": "83", "0x42": "2c", "0x43": "1a", "0x44": "1b", "0x45": "6e", "0x46": "5a", "0x47": "a0",
  "0x48": "52", "0x49": "3B", "0x4a": "d6", "0x4b": "b3", "0x4c": "29", "0x4d": "e3", "0x4e": "2f", "0x4f": "84",

  "0x50": "53", "0x51": "d1", "0x52": "00", "0x53": "ed", "0x54": "20", "0x55": "fc", "0x56": "b1", "0x57": "5b",
  "0x58": "6a", "0x59": "cb", "0x5a": "be", "0x5b": "39", "0x5c": "4a", "0x5d": "4c", "0x5e": "58", "0x5f": "cf",

  "0x60": "d0", "0x61": "ef", "0x62": "aa", "0x63": "fb", "0x64": "43", "0x65": "4d", "0x66": "33", "0x67": "85",
  "0x68": "45", "0x69": "f9", "0x6a": "02", "0x6b": "7f", "0x6c": "50", "0x6d": "3c", "0x6e": "9f", "0x6f": "a8",

  "0x70": "51", "0x71": "a3", "0x72": "40", "0x73": "8f", "0x74": "92", "0x75": "9d", "0x76": "38", "0x77": "f5",
  "0x78": "bc", "0x79": "b6", "0x7a": "da", "0x7b": "21", "0x7c": "10", "0x7d": "ff", "0x7e": "f3", "0x7f": "d2",

  "0x80": "cd", "0x81": "0c", "0x82": "13", "0x83": "ec", "0x84": "5f", "0x85":"97" , "0x86": "44", "0x87": "17", 
  "0x88": "c4", "0x89": "a7", "0x8a": "7e", "0x8b": "3d", "0x8c": "64", "0x8d": "5d", "0x8e": "19", "0x8f": "73",

  "0x90": "60", "0x91": "81", "0x92": "4f", "0x93": "dc", "0x94": "22", "0x95": "2a", "0x96": "90", "0x97": "88", 
  "0x98": "46", "0x99": "ee", "0x9a": "b8", "0x9b": "14", "0x9c": "de", "0x9d": "5e", "0x9e": "0b", "0x9f": "db",

  "0xa0": "e0", "0xa1": "32", "0xa2": "3a", "0xa3": "0a", "0xa4": "49", "0xa5": "06", "0xa6": "24", "0xa7": "5c",
  "0xa8": "c2", "0xa9": "d3", "0xaa": "ac", "0xab": "62", "0xac": "91", "0xad": "95", "0xae": "e4", "0xaf": "79",

  "0xb0": "e7", "0xb1": "c8", "0xb2": "37", "0xb3": "6d", "0xb4": "8d", "0xb5": "d5", "0xb6": "4e", "0xb7": "a9",
  "0xb8": "6c", "0xb9": "56", "0xba": "f4", "0xbb": "ea", "0xbc": "65", "0xbd": "7a", "0xbe": "ae", "0xbf": "08",

  "0xc0": "ba", "0xc1": "78", "0xc2": "25", "0xc3": "2e", "0xc4": "1c", "0xc5": "a6", "0xc6": "b4", "0xc7": "c6",
  "0xc8": "e8", "0xc9": "dd", "0xca": "74", "0xcb": "1f", "0xcc": "4b", "0xcd": "bd", "0xce": "8b", "0xcf": "8a",

  "0xd0": "70", "0xd1": "3e", "0xd2": "b5", "0xd3": "66", "0xd4": "48", "0xd5": "03", "0xd6": "f6", "0xd7": "0e",
  "0xd8": "61", "0xd9": "35", "0xda": "57", "0xdb": "b9", "0xdc": "86", "0xdd": "c1", "0xde": "1d", "0xdf": "9e",
  
  "0xe0": "e1", "0xe1": "f8", "0xe2": "98", "0xe3": "11", "0xe4": "69", "0xe5": "d9", "0xe6": "8e", "0xe7": "94",
  "0xe8": "9b", "0xe9": "1e", "0xea": "87", "0xeb": "e9", "0xec": "ce", "0xed": "55", "0xee": "28", "0xef": "df",
 
  "0xf0": "8c", "0xf1": "a1", "0xf2": "89", "0xf3": "0d", "0xf4": "bf", "0xf5": "e6", "0xf6": "42", "0xf7": "68",
  "0xf8": "41", "0xf9": "99", "0xfa": "2f", "0xfb": "0f", "0xfc": "b0", "0xfd": "54", "0xfe": "bb", "0xff": "16",
}

time: 7.47 ms (started: 2022-09-21 19:50:25 +00:00)


In [ ]:
keys = np.random.randint(5,100,50)
print(keys)

[29 47 33 65 49 75 15 46 80  6 37 73 46 28 78 50 81 28 98 61 53 55 83 55
 67 18 64  6 38  6 96  8  7 58 98 41 88 87 46 67 17 98 99 39 95 78 51 14
 25 32]
time: 1.69 ms (started: 2022-09-21 19:53:19 +00:00)


In [ ]:

Message = "This is our Algorithm"
print("The message Alice wants to send Bob is: "+Message)
M = [*Message] #  *** converting message to a list of character ***

print(" ")
print("The set of key used for encryption and decyption is generated through D-H and Collatz Conjecture")
print(" ")

keys = [7,7,18,8,104,23,20,20,104,103,25,110,27,5,28,22,105,17,25,22,6]
print("Secret Key = [7,7,18,8,104,23,20,20,104,103,25,110,27,5,28,22,105,17,25,22,6]")
print(" ")



# <Check if the key is a prime>
def perfectSquareCheck(num):

    root = math.sqrt(num)
    return int(root + 0.5) ** 2 == num

# <Check if the key is a perfect square>     
def isPrimeCheck(num):
  flag = True
  if num > 1:
      for i in range(2, num):
          if (num % i) == 0:
              flag = False
              break
  return flag

# <Check if the key is a prime>  
def consecPrimeCheck(currNum,nextNum):  
   return isPrimeCheck(currNum) and isPrimeCheck(nextNum)





The message Alice wants to send Bob is: This is our Algorithm
 
The set of key used for encryption and decyption is generated through D-H and Collatz Conjecture
 
Secret Key = [7,7,18,8,104,23,20,20,104,103,25,110,27,5,28,22,105,17,25,22,6]
 
time: 10.1 ms (started: 2022-09-21 19:54:44 +00:00)


In [ ]:

# Encryption on Plaintext using the Secret key stream

print("Alice will Encrypt the Plaintext using the Secret key stream")
print(" ")

def encryption(M,keys):
  cipherText = ""
  for i in range(len(M)):
   keyIndex = i % len(keys)
   
   # *** Shift Cipher ***
   if consecPrimeCheck(keys[keyIndex],keys[(keyIndex+1)%len(keys)]) == True:
    
    cipherText = cipherText + chr(ord(M[i]) + keys[keyIndex]) 
   
   # *** Byte Xor ***
   elif perfectSquareCheck(keys[keyIndex]) == True:
    
    cipherText = cipherText + chr(ord(M[i])^keys[keyIndex])   
    
   # *** Byte Injection ***
   elif isPrimeCheck(keys[keyIndex]) == True:
    
    cipherText = cipherText + str(M[i]) + chr(2*keys[keyIndex] + 128)  
    
   # *** Byte Substitution ***
   else : 
    S = (ord(M[i]) + keys[keyIndex])
    #print(S)
    A = hex(S % 256)
    #print(A)
    if len(A)==3:
     A =   A + "0" 
     #print(A)
    sbox_value = S_BOX[A]                                   
    cipherText = cipherText + chr(int(sbox_value,16) + 128)
    #print(cipherText)
  return cipherText
  
print("Cipertext sent to Bob: "+ encryption(Message,keys))
print(" ")

Alice will Encrypt the Plaintext using the Secret key stream
 
Cipertext sent to Bob: [h¡¡ńi®uŎkÊlŬėĹi¢mųď
 
time: 4.2 ms (started: 2022-09-21 19:54:59 +00:00)


In [ ]:
Ciphertext = encryption(M,keys)
print("Bob receives encrypted message: "+encryption(M,keys)) #  *** converting message to a list of character ***
print(" ")
C = [[*Ciphertext]]


def Decryption(C,keys):
  tempText = ""
  j = 0
  i = 0
  while j < len(C):
   keyIndex = i % len(keys)

   # *** Shift Cipher ***
   if consecPrimeCheck(keys[keyIndex],keys[(keyIndex+1)%len(keys)]) == True:
    
     tempText = tempText + chr(ord(C[j]) - keys[keyIndex]) 

   # *** Byte Xor ***
   elif perfectSquareCheck(keys[keyIndex]) == True:
     
     tempText = tempText + chr(keys[keyIndex]^ord(C[j])) 

   # *** Byte Deletion ***
   elif isPrimeCheck(keys[keyIndex]) == True:
    
     tempText = tempText + str(C[j])  
     j = j+1
 
 
   # *** Byte Substitution ***
   else:
     new_value = ""  
     temp = ord(C[j]) - 128
     key_value = hex(temp)
     #print(key_value)
                                           
     for z in range(2,len(key_value)):
       new_value+= key_value[z]
     #print(new_value)
     #myVal = "" 
     if len(new_value) == 1:
       new_value = "0"+new_value
     #print(new_value)
     myVal = "" 
     value = {x for x in S_BOX if S_BOX[x]==new_value}
     for y in value:
       myVal+=y
     #print(myVal)
     if myVal =="0x4e0xfa":
      tempText = tempText + "," 
     else:
      tempText = tempText + chr((int(myVal, 16) - keys[keyIndex])) 
   j = j+1
   i = i+1
  return tempText

print("Bob retrieves Plaintext : "+ Decryption(Ciphertext,keys))
print(" ")



Bob receives encrypted message: [h¡¡ńi®uŎkÊlŬėĹi¢mųď
 
Bob retrieves Plaintext : This is our Algorithm
 
time: 10.9 ms (started: 2022-09-21 19:55:06 +00:00)


In [ ]:
#keys = np.random.randint(5,100,200)